In [30]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

In [31]:
df = df[df.osoba.isin(['Kamil','AK','J'])]

In [32]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['date'] = timestamp_pl.dt.date
df.sample(2)


,timestamp,duration,app,status,device,osoba,category,timestamp_pl,date
75097,2026-01-07 21:55:09.226000+00:00,5.067,steamwebhelper.exe,not-afk,PC,Kamil,gry komputerowe,2026-01-07 22:55:09.226000+01:00,2026-01-07
47871,2025-12-29 21:20:13.548000+00:00,1.019,Discord.exe,not-afk,PC,J,Komunikatory,2025-12-29 22:20:13.548000+01:00,2025-12-29


In [33]:
Suma_z_dnia = df.groupby(['osoba','date'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] = round(Suma_z_dnia['duration'] / 3600,2)
Suma_z_dnia = Suma_z_dnia.sort_values(['osoba','date'])
Suma_z_dnia.head()

,osoba,date,duration
0,AK,2025-12-13,0.98
1,AK,2025-12-14,12.11
2,AK,2025-12-15,5.16
3,AK,2025-12-17,3.07
4,AK,2025-12-18,1.98


In [34]:
# Wszystkie dni z zakresu
osoby = Suma_z_dnia['osoba'].unique()
daty = pd.date_range(Suma_z_dnia['date'].min(), Suma_z_dnia['date'].max())

full_index = pd.MultiIndex.from_product([osoby, daty], names=['osoba', 'date'])
Suma_z_dnia = Suma_z_dnia.set_index(['osoba', 'date']).reindex(full_index, fill_value=0).reset_index()
Suma_z_dnia.sort_values(["osoba","date"]).reset_index(drop=True)

,osoba,date,duration
0,AK,2025-12-03,0.00
1,AK,2025-12-04,0.00
2,AK,2025-12-05,0.00
3,AK,2025-12-06,0.00
4,AK,2025-12-07,0.00
...,...,...,...
112,Kamil,2026-01-06,3.29
113,Kamil,2026-01-07,3.83
114,Kamil,2026-01-08,5.17
115,Kamil,2026-01-09,7.46


In [48]:
# zakładam, że df['date'] jest typu datetime
Suma_z_dnia['duration_hm'] = Suma_z_dnia['duration'].apply(lambda x: f"{int(x)}h {int((x%1*60))}m")


fig = px.line(
    Suma_z_dnia,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności', 'osoba':'Osoba'},
    title='Dzienna aktywność',
    template='plotly_white',
    line_shape='spline',
        hover_data={
        'duration': False,       # ukrywamy dziesiętne godziny
        'duration_hm': True,     # pokazujemy Hh Mm
        'osoba': True
    }
)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_yaxes(ticksuffix="h")
fig.update_layout(xaxis_tickangle=-45)  # dla czytelności etykiet dat
fig.update_traces(line=dict(width=4))  # np. 4 piksele

start = Suma_z_dnia['date'].min()
end = Suma_z_dnia['date'].max()
tick_vals = pd.date_range(start, end, freq='2D')  # co 2 dni
fig.update_xaxes(tickvals=tick_vals, tickformat="%Y-%m-%d")

for trace in fig.data:
    osoba = trace.name  # nazwa linii = osoba
    # filtrujemy dane tylko dla tej osoby
    df_osoba = Suma_z_dnia[Suma_z_dnia['osoba'] == osoba]
    
    # przypisujemy customdata dla tej linii
    trace.customdata = df_osoba[['duration_hm']].values
    
    # własny hovertemplate
    trace.hovertemplate = (
        f"<b>{osoba}</b><br>"       # nazwa osoby
        "Data: %{x|%Y-%m-%d}<br>"   # data
        "Czas aktywności: %{customdata[0]}<extra></extra>"  # Hh Mm
    )


fig.show()


In [16]:
# zakładam, że df['date'] jest typu datetime
fig = px.bar(
    Suma_z_dnia,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna',
           template='plotly_white',

)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_yaxes(ticksuffix="h")
fig.update_layout(xaxis_tickangle=-45,  barmode='group')  # dla czytelności etykiet dat

fig.show()


In [18]:
osoba = 'J'
Suma_osoby = Suma_z_dnia[Suma_z_dnia.osoba == osoba]

fig = px.bar(
    Suma_osoby,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna',
           template='plotly_white',

)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_layout(xaxis_tickangle=-45,  barmode='group')  # dla czytelności etykiet dat
fig.update_yaxes(ticksuffix="h")

fig.show()
